In [2]:
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error

import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [1]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-experiment")

c:\Users\heza\Documents\Projectz\MLOps-Zoomcamp-zachho9\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1790650640967, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790650640967, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

### Q1. Downloading the data



In [3]:
df = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet')

In [4]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2023-03-01 00:06:43,2023-03-01 00:16:43,1.0,0.00,1.0,N,238,42,2,8.6,1.0,0.5,0.00,0.0,1.0,11.10,0.0,0.00
1,2,2023-03-01 00:08:25,2023-03-01 00:39:30,2.0,12.40,1.0,N,138,231,1,52.7,6.0,0.5,12.54,0.0,1.0,76.49,2.5,1.25
2,1,2023-03-01 00:15:04,2023-03-01 00:29:26,0.0,3.30,1.0,N,140,186,1,18.4,3.5,0.5,4.65,0.0,1.0,28.05,2.5,0.00
3,1,2023-03-01 00:49:37,2023-03-01 01:01:05,1.0,2.90,1.0,N,140,43,1,15.6,3.5,0.5,4.10,0.0,1.0,24.70,2.5,0.00
4,2,2023-03-01 00:08:04,2023-03-01 00:11:06,1.0,1.23,1.0,N,79,137,1,7.2,1.0,0.5,2.44,0.0,1.0,14.64,2.5,0.00


In [5]:
len(df)

3403766

In [6]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [7]:
df = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet')
print(len(df))

3316216


### Q2. Computing duration

In [8]:
df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
df['duration'] = df.duration.dt.total_seconds() / 60

### Q4. One-hot encoding

In [9]:
categorical = ['PULocationID', 'DOLocationID']

In [10]:
df[categorical] = df[categorical].astype(str)

In [11]:
train_dicts = df[categorical].to_dict(orient='records')

In [12]:
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

In [14]:
print(f'Feature matrix size: {X_train.shape}')

Feature matrix size: (3316216, 518)


### Q5. Training a model

In [15]:
target = 'duration'
y_train = df[target].values

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_train)

rmse = root_mean_squared_error(y_train, y_pred)

In [ ]:
with mlflow.start_run():
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("lr_intercept", float(lr.intercept_))
    mlflow.sklearn.log_model(lr, artifact_path="linear_regression_model")
    

2026/09/30 09:41:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/30 09:41:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run bouncy-snipe-944 at: http://localhost:5000/#/experiments/1/runs/49e33b9b7e9c42e99a1605621a0157c9
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [21]:
from mlflow.models import Model

# Load the target model's configuration data
model_config = Model.load("runs:/49e33b9b7e9c42e99a1605621a0157c9/linear_regression_model")

# Safely check if model_size_bytes exists and is populated
size_bytes = getattr(model_config, "model_size_bytes", None)

if size_bytes is not None:
    print(f"Model Size: {size_bytes} bytes")
    print(f"Model Size: {size_bytes / (1024**2):.2f} MB")
else:
    print("Model size was not calculated or stored when this model was saved.")

Model Size: 8628 bytes
Model Size: 0.01 MB
